# Pancancer enrichment analysis step 17: Visualize results from rSEA

## Setup

In [1]:
import os
import datetime
import altair as alt
import numpy as np
import pandas as pd
import operator
import IPython.display
from toolz.curried import pipe

In [13]:
TIME_START = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')

# Step 1 output
STEP01_DIR = "step01_outputs"
STEP01_FILE = "tumor_change_20200529_195104_10000_perm.tsv"
STEP01_FILE_PATH = os.path.join(STEP01_DIR, STEP01_FILE)

# Step 16 output
STEP16_DIR = "step16_outputs"
STEP16_thresh01 = os.path.join(STEP16_DIR, "enrichment_rsea_20200626_114446_thresh_0.1.tsv")

# Reactome .gmt file
REACTOME_GMT = os.path.join("gene_set_libraries", "Reactome_2020.gmt")

MSV = 0.5 # Simplified expression value for marginally significant proteins

In [3]:
# Altair options
alt.data_transformers.disable_max_rows()

def json_dir(data, data_dir):
    os.makedirs(data_dir, exist_ok=True)
    return pipe(data, alt.to_json(filename=os.path.join(data_dir, "{prefix}-{hash}.{extension}")) )

alt.data_transformers.register("json_dir", json_dir)
alt.data_transformers.enable("json_dir", data_dir="step17_altair_data")

DataTransformerRegistry.enable('json_dir')

## Read in data from step 16

In [4]:
enrichment_data = pd.read_csv(STEP16_thresh01, sep="\t", index_col=0)
enrichment_data.columns = enrichment_data.columns.str.replace(".", "_", regex=False)
enrichment_data

,Name,Size,Coverage,TDP_bound,TDP_estimate,SC_adjP,Comp_adjP,Comp_0_5_adjP,cancer_type
ID,,,,,,,,,
1,R-HSA-164843,13,0.69,0.444444,0.444444,0.0,1.000000,1.000000,ccrcc
2,R-HSA-73843,3,0.67,0.500000,0.500000,0.0,1.000000,1.000000,ccrcc
3,R-HSA-1971475,26,0.77,0.500000,0.500000,0.0,1.000000,1.000000,ccrcc
4,R-HSA-5619084,77,0.86,0.803030,0.818182,0.0,0.000000,0.000000,ccrcc
5,R-HSA-1369062,18,0.56,0.400000,0.400000,0.0,1.000000,1.000000,ccrcc
6,R-HSA-382556,103,0.84,0.712644,0.724138,0.0,0.000000,0.000000,ccrcc
7,R-HSA-9033807,3,0.33,0.000000,0.000000,1.0,1.000000,1.000000,ccrcc
8,R-HSA-9660821,134,0.31,0.666667,0.666667,0.0,0.504000,0.000000,ccrcc
9,R-HSA-418592,25,0.84,0.666667,0.666667,0.0,1.000000,0.000000,ccrcc


In [5]:
alt.Chart(enrichment_data).mark_bar().encode(
    x=alt.X("Comp_adjP", bin=alt.Bin(step=0.05)),
    y="count()",
)

alt.Chart(...)

In [6]:
alt.Chart(enrichment_data).mark_bar().encode(
    x=alt.X("SC_adjP", bin=alt.Bin(step=0.05)),
    y="count()",
)

alt.Chart(...)

In [7]:
(enrichment_data["Comp_adjP"] == 0).sum()

7351

In [8]:
(enrichment_data["SC_adjP"] == 0).sum()

16174

## Figure 1: Bubble plot

In [48]:
def plot_top_ten(enrich_file_path, expr_file_path, gmt_file_path, xtitle, min_size, sort_col, sort_asc, cutoff_col, cutoff=0.05):
    
    # Read in the expression data
    all_expression_data = pd.read_csv(expr_file_path, sep="\t", index_col=0)

    # Make a column where all increases are +1 and all decreases 
    # are -1, because these are ratioed abundances, so we can't 
    # compare magnitudes between genes--we can only compare whether 
    # there was a change, and whether it was positive or negative
    all_expression_data = all_expression_data.assign(simplified_change=np.nan)

    # adj p < 0.05 and change > 1 => +1
    all_expression_data.loc[
        (all_expression_data["change"] > 0) & (all_expression_data["adj_p"] < 0.05), 
        "simplified_change"
    ] = 1

    # adj p >= 0.05 and change > 1 => +0.5
    all_expression_data.loc[(all_expression_data["change"] > 0) & (all_expression_data["adj_p"] >= 0.05),
        "simplified_change"
    ] = MSV

    # change == 0 => 0
    all_expression_data.loc[
        all_expression_data["change"] == 0,
        "simplified_change"
    ] = 0

    # adj p >= 0.05 and change < 1 => -0.5
    all_expression_data.loc[(all_expression_data["change"] < 0) & (all_expression_data["adj_p"] >= 0.05), 
        "simplified_change"
    ] = -MSV

    # adj p < 0.05 and change < 1 => -1
    all_expression_data.loc[
        (all_expression_data["change"] < 0) & (all_expression_data["adj_p"] < 0.05),
        "simplified_change"
    ] = -1

    # Select just the proteins where we chose to reject the null hypothesis of no change
    expression_data = all_expression_data[all_expression_data["reject_null"]]
    
    # Read in the GMT file and join in pathway names and gene lists
    with open(gmt_file_path, "r") as fp:
        gene_lists = fp.readlines()

    # Take the newline off the end, and split on the tab character to create a list of lists
    gene_lists = [l.strip().split("\t") for l in gene_lists]

    # Create a dataframe mapping pathway ID to pathway name and genes
    pathway_ids = [l[1] for l in gene_lists]
    pathway_names = [l[0] for l in gene_lists]
    pathway_genes = [l[2:] for l in gene_lists]
    pathway_data = pd.DataFrame({
        "pathway_id": pathway_ids,
        "pathway_name": pathway_names,
        "pathway_genes": pathway_genes
    })

    # Read in the enrichment data
    enrichment_data = pd.read_csv(enrich_file_path, sep="\t", index_col=0)
    enrichment_data.columns = enrichment_data.columns.str.replace(".", "_", regex=False)
    enrichment_data = enrichment_data.rename(columns={"Name": "pathway_id"})
    
    # Merge pathway data into the enrichment data
    enrichment_data = enrichment_data.merge(
        pathway_data,
        how="left",
        left_on="pathway_id",
        right_on="pathway_id",
        validate="many_to_one"
    )
    
    # Select pathways that meet the minimum size
    enrichment_data = enrichment_data[
        enrichment_data["pathway_genes"].apply(len) >= min_size
    ]

    # Select enrichment data below our cutoff
    enrichment_data = enrichment_data.loc[enrichment_data[cutoff_col] <= cutoff, :]
    
    # Assign pathway ranks within each cancer type based on the sort_col. Make a custom one if needed.
    if sort_col != cutoff_col:
        if sort_asc:
            enrichment_data = enrichment_data.assign(
                temp_sort=(enrichment_data[sort_col] + 0.1) * (enrichment_data[cutoff_col] + 0.1)
            )
        else:
            enrichment_data = enrichment_data.assign(
                temp_sort=(enrichment_data[sort_col] + 0.1) / (enrichment_data[cutoff_col] + 0.1)
            )
            
        sort_col = "temp_sort"
    
    enrichment_data = enrichment_data.\
        assign(cancer_rank=enrichment_data.groupby("cancer_type")[sort_col].rank(ascending=sort_asc)).\
        sort_values(by=["cancer_type", "cancer_rank"]).\
        reset_index(drop=True)

    # Make a table with summary info for all pathways
    enrichment_summary = enrichment_data[["pathway_id"]].drop_duplicates(keep="first")

    pathway_times_enriched = enrichment_summary["pathway_id"].apply(
        lambda x: enrichment_data[enrichment_data["pathway_id"] == x].shape[0])

    avg_rank = enrichment_summary["pathway_id"].apply(
        lambda x: enrichment_data.loc[enrichment_data["pathway_id"] == x, "cancer_rank"].mean())

    enrichment_summary = enrichment_summary.\
        assign(
            pathway_times_enriched=pathway_times_enriched,
            pathway_avg_rank=avg_rank).\
        sort_values(
            by=["pathway_times_enriched", "pathway_avg_rank"],
            ascending=[False, True]).\
        reset_index(drop=True)

    # Merge in the original enrichment data
    enrichment_data = enrichment_data.\
        merge(
            enrichment_summary,
            how="outer",
            left_on="pathway_id",
            right_on="pathway_id",
            validate="many_to_one"
        ).\
        sort_values(
            by=["pathway_times_enriched", "pathway_avg_rank", "cancer_type"],
            ascending=[False, True, True]
        )

    # Select top 10 for our plot
    in_all = enrichment_summary.loc[
        enrichment_summary["pathway_times_enriched"] == 8,
        "pathway_id"
    ]
    
    if in_all.size <= 10:
        top_ten = in_all
    else:
        top_ten = in_all[:10]
    
    sel_enrich = enrichment_data[enrichment_data["pathway_id"].isin(top_ten)]

    # Calculate the mean expression for each pathway in each cancer type
    mean_exprs = []

    for idx in sel_enrich.index:
        genes = sel_enrich.loc[idx, "pathway_genes"]
        cancer_type = sel_enrich.loc[idx, "cancer_type"]

        genes_expr = expression_data.loc[
            expression_data["protein_str"].isin(genes),
            "simplified_change"
        ].mean()

        mean_exprs.append(genes_expr)

    sel_enrich = sel_enrich.assign(mean_expr=mean_exprs)

    sel_enrich = sel_enrich.assign(
        rank_size=1 / sel_enrich["cancer_rank"],
        avg_rank_size=1 / sel_enrich["pathway_avg_rank"],
        avg_rank_label=sel_enrich["pathway_avg_rank"].apply(round, 2).astype(str))
    
    # Take care of duplicates for the upper plot
    for i in range(10):
        sel_enrich["avg_rank_label"] = sel_enrich["avg_rank_label"].where(
            cond=~(sel_enrich.duplicated(subset=["cancer_type", "avg_rank_label"], keep="first")),
            other=" " + sel_enrich["avg_rank_label"])

    individual = alt.Chart(sel_enrich).mark_circle().encode(
        x=alt.X(
            "pathway_name:N",
            sort=sel_enrich["pathway_name"].values,
            axis=alt.Axis(
                labelAngle=-30,
                labelFontSize=12,
                labelLimit=500,
                title="",
                titleFontSize=16
            )
        ),
        y=alt.Y(
            "cancer_type:N",
            axis=alt.Axis(
                title="Cancer type",
                titleFontSize=12
            ),
        ),
        size=alt.Size(
            "rank_size:Q",
            legend=None
        ),
        color=alt.Color(
            "mean_expr:Q",
            scale=alt.Scale(
                scheme="blueorange",
                domain=[-1, 1]
            ),
            legend=alt.Legend(
                title="Pathway tumor expression"
            )
        )
    ).properties(
        width=400,
        height=300
    )

    aggregate = alt.Chart(sel_enrich).mark_circle().encode(
        x=alt.X(
            "avg_rank_label:N",
            sort=sel_enrich["avg_rank_label"].values,
            axis=alt.Axis(
                labelAngle=-30,
                labelFontSize=12,
                labelLimit=500,
                title="Overall rank of pathway",
                titleFontSize=12
            )
        ),
        size=alt.Size(
            "avg_rank_size:Q",
            legend=None
        ),
    ).properties(
        width=400
    )

    full_plot = alt.vconcat(
        aggregate, individual
    ).properties(
        title=xtitle
    )
    
    return full_plot, enrichment_data, all_expression_data, enrichment_summary

In [52]:
alt.vconcat(
    plot_top_ten(
        enrich_file_path=STEP16_thresh01, 
        expr_file_path=STEP01_FILE_PATH, 
        gmt_file_path=REACTOME_GMT,
        xtitle="Reactome data - rSEA, threshold = 1, sort by Coverage / Comp_adjP, cutoff Comp_adjP",
        min_size=5,
        sort_col="Coverage",
        sort_asc=False,
        cutoff_col="Comp_adjP",
        cutoff=0.05
    )[0],
    plot_top_ten(
        enrich_file_path=STEP16_thresh01, 
        expr_file_path=STEP01_FILE_PATH, 
        gmt_file_path=REACTOME_GMT,
        xtitle="Reactome data - rSEA, threshold = 1, sort by Coverage / SC_adjP, cutoff SC_adjP",
        min_size=5,
        sort_col="Coverage",
        sort_asc=False,
        cutoff_col="SC_adjP",
        cutoff=0.05
    )[0],
).configure_axis(
    grid=True
).configure_title(
    fontSize=16,
    anchor="end",
    offset=20
).resolve_scale(
    size="independent"
)

alt.VConcatChart(...)

In [55]:
e = plot_top_ten(
    enrich_file_path=STEP16_thresh01, 
    expr_file_path=STEP01_FILE_PATH, 
    gmt_file_path=REACTOME_GMT,
    xtitle="Reactome data - rSEA, threshold = 1, sort by Coverage, cutoff Comp_adjP",
    min_size=5,
    sort_col="Coverage",
    sort_asc=False,
    cutoff_col="SC_adjP",
    cutoff=0.05
)[1]

In [57]:
e

,pathway_id,Size,Coverage,TDP_bound,TDP_estimate,SC_adjP,Comp_adjP,Comp_0_1_adjP,cancer_type,pathway_name,pathway_genes,temp_sort,cancer_rank,pathway_times_enriched,pathway_avg_rank
256,R-HSA-372708,15,1.20,0.500000,0.500000,0.0,1.000000,0.0,ccrcc,p130Cas linkage to MAPK signaling for integrins,"[APBB1IP, BCAR1, CRK, FGA, FGB, FGG, FN1, ITGA...",13.0,27.5,8,51.1875
257,R-HSA-372708,15,1.00,0.466667,0.466667,0.0,1.000000,0.0,colon,p130Cas linkage to MAPK signaling for integrins,"[APBB1IP, BCAR1, CRK, FGA, FGB, FGG, FN1, ITGA...",11.0,29.0,8,51.1875
258,R-HSA-372708,15,1.00,0.800000,0.800000,0.0,0.000000,0.0,endometrial,p130Cas linkage to MAPK signaling for integrins,"[APBB1IP, BCAR1, CRK, FGA, FGB, FGG, FN1, ITGA...",11.0,79.5,8,51.1875
259,R-HSA-372708,15,1.00,0.800000,0.800000,0.0,0.000000,0.0,gbm,p130Cas linkage to MAPK signaling for integrins,"[APBB1IP, BCAR1, CRK, FGA, FGB, FGG, FN1, ITGA...",11.0,84.5,8,51.1875
260,R-HSA-372708,15,1.00,0.533333,0.533333,0.0,0.989873,0.0,hnscc,p130Cas linkage to MAPK signaling for integrins,"[APBB1IP, BCAR1, CRK, FGA, FGB, FGG, FN1, ITGA...",11.0,115.5,8,51.1875
261,R-HSA-372708,15,1.20,0.777778,0.777778,0.0,0.000000,0.0,lscc,p130Cas linkage to MAPK signaling for integrins,"[APBB1IP, BCAR1, CRK, FGA, FGB, FGG, FN1, ITGA...",13.0,21.0,8,51.1875
262,R-HSA-372708,15,1.13,0.823529,0.823529,0.0,0.000000,0.0,luad,p130Cas linkage to MAPK signaling for integrins,"[APBB1IP, BCAR1, CRK, FGA, FGB, FGG, FN1, ITGA...",12.3,26.5,8,51.1875
263,R-HSA-372708,15,1.27,0.578947,0.578947,0.0,0.000000,0.0,ovarian,p130Cas linkage to MAPK signaling for integrins,"[APBB1IP, BCAR1, CRK, FGA, FGB, FGG, FN1, ITGA...",13.7,26.0,8,51.1875
0,R-HSA-264870,12,2.00,0.708333,0.750000,0.0,0.000000,0.0,ccrcc,Caspase-mediated cleavage of cytoskeletal prot...,"[ADD1, CASP3, CASP6, CASP7, CASP8, DBNL, GAS2,...",21.0,1.0,8,51.8125
1,R-HSA-264870,12,0.92,0.545455,0.636364,0.0,0.254300,0.0,colon,Caspase-mediated cleavage of cytoskeletal prot...,"[ADD1, CASP3, CASP6, CASP7, CASP8, DBNL, GAS2,...",10.2,130.0,8,51.8125


In [56]:
e[e["pathway_name"] == "Neutrophil degranulation"]

,pathway_id,Size,Coverage,TDP_bound,TDP_estimate,SC_adjP,Comp_adjP,Comp_0_1_adjP,cancer_type,pathway_name,pathway_genes,temp_sort,cancer_rank,pathway_times_enriched,pathway_avg_rank
3534,R-HSA-6798695,479,0.93,0.720358,0.751678,0.0,0.000000,0.0,ccrcc,Neutrophil degranulation,"[A1BG, ABCA13, ACAA1, ACLY, ACP3, ACTR10, ACTR...",10.3,453.0,8,396.4375
3535,R-HSA-6798695,479,0.85,0.538272,0.617284,0.0,0.448200,0.0,colon,Neutrophil degranulation,"[A1BG, ABCA13, ACAA1, ACLY, ACP3, ACTR10, ACTR...",9.5,301.5,8,396.4375
3536,R-HSA-6798695,479,0.89,0.491803,0.566745,0.0,0.194133,0.0,endometrial,Neutrophil degranulation,"[A1BG, ABCA13, ACAA1, ACLY, ACP3, ACTR10, ACTR...",9.9,525.5,8,396.4375
3537,R-HSA-6798695,479,0.90,0.506944,0.601852,0.0,0.711143,0.0,gbm,Neutrophil degranulation,"[A1BG, ABCA13, ACAA1, ACLY, ACP3, ACTR10, ACTR...",10.0,488.5,8,396.4375
3538,R-HSA-6798695,479,0.94,0.529018,0.589286,0.0,0.074486,0.0,hnscc,Neutrophil degranulation,"[A1BG, ABCA13, ACAA1, ACLY, ACP3, ACTR10, ACTR...",10.4,402.0,8,396.4375
3539,R-HSA-6798695,479,0.94,0.659292,0.725664,0.0,0.747742,0.0,lscc,Neutrophil degranulation,"[A1BG, ABCA13, ACAA1, ACLY, ACP3, ACTR10, ACTR...",10.4,262.0,8,396.4375
3540,R-HSA-6798695,479,0.91,0.635945,0.714286,0.0,0.386400,0.0,luad,Neutrophil degranulation,"[A1BG, ABCA13, ACAA1, ACLY, ACP3, ACTR10, ACTR...",10.1,284.5,8,396.4375
3541,R-HSA-6798695,479,0.94,0.265487,0.362832,0.0,0.229047,0.0,ovarian,Neutrophil degranulation,"[A1BG, ABCA13, ACAA1, ACLY, ACP3, ACTR10, ACTR...",10.4,454.5,8,396.4375
